# SPHEREx two-epoch image analysis

This notebook starts the scientific analysis of two SPHEREx calibrated FITS cutouts. It visualizes the IMAGE extensions, checks metadata/WCS, detects candidate sources, and exports measured sky positions. **Detections are not confirmed moving objects.** Image registration, cross-matching, uncertainty checks and validation are required before claiming motion.

Upload both original `.fits` files when prompted. Do not rename or modify the originals.


In [ ]:
!pip -q install astropy photutils
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
from google.colab import files

uploaded = files.upload()
fits_files = [name for name in uploaded if name.lower().endswith(('.fits','.fit','.fits.gz'))]
print('FITS files:', fits_files)
if len(fits_files) != 2:
    print('Please upload exactly the two SPHEREx FITS files for this first comparison.')


In [ ]:
def inspect_fits(path):
    hdul = fits.open(path)
    print('\nFILE:', path)
    hdul.info()
    for i, hdu in enumerate(hdul):
        if hdu.data is not None:
            print(i, hdu.name, 'shape=', hdu.data.shape, 'BUNIT=', hdu.header.get('BUNIT'),
                  'DATE-OBS=', hdu.header.get('DATE-OBS'), 'MJD-OBS=', hdu.header.get('MJD-OBS'))
    return hdul

hduls = [inspect_fits(f) for f in fits_files]


In [ ]:
def get_image_hdu(hdul):
    # Prefer the named IMAGE extension; fall back only to a 2-D image array.
    for hdu in hdul:
        if hdu.name.upper() == 'IMAGE' and hdu.data is not None and np.asarray(hdu.data).ndim == 2:
            return hdu
    for hdu in hdul:
        if hdu.data is not None and np.asarray(hdu.data).ndim == 2:
            return hdu
    raise ValueError('No 2-D image extension found')

image_hdus = [get_image_hdu(h) for h in hduls]
images = [np.asarray(h.data, dtype=float) for h in image_hdus]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, img, name in zip(axes, images, fits_files):
    finite = np.isfinite(img)
    lo, hi = np.nanpercentile(img[finite], [2, 99.5])
    ax.imshow(img, origin='lower', cmap='gray', vmin=lo, vmax=hi)
    ax.set_title(name, fontsize=8)
    ax.set_xlabel('Pixel x'); ax.set_ylabel('Pixel y')
plt.tight_layout(); plt.show()


In [ ]:
# Inspect WCS availability and image geometry. WCS may be stored in IMAGE or another header.
for name, hdu, img in zip(fits_files, image_hdus, images):
    print('\n', name, 'image shape:', img.shape)
    print('WCS-related header cards:')
    for key in ['CTYPE1','CTYPE2','CRPIX1','CRPIX2','CRVAL1','CRVAL2','CDELT1','CDELT2','CD1_1','CD1_2','CD2_1','CD2_2']:
        if key in hdu.header: print(key, '=', hdu.header[key])
    try:
        w = WCS(hdu.header)
        print(w)
    except Exception as e:
        print('Could not construct WCS from IMAGE header:', e)


## Initial source detection

This is a first-pass detector only. It uses a local background/noise estimate and DAOStarFinder. Crowded fields, extended sources, artifacts and SPHEREx-specific PSF/quality flags may require a more tailored pipeline. Review the plotted detections before using the CSV.


In [ ]:
tables = []
for idx, (name, hdul, hdu, img) in enumerate(zip(fits_files, hduls, image_hdus, images), start=1):
    mean, median, std = sigma_clipped_stats(img, sigma=3.0, maxiters=5)
    if not np.isfinite(std) or std <= 0:
        print('Skipping source detection for', name, ': invalid noise estimate')
        continue
    finder = DAOStarFinder(fwhm=3.0, threshold=5.0*std)
    sources = finder(img - median)
    if sources is None:
        print('No sources detected at this threshold in', name)
        continue
    df = sources.to_pandas()
    df['observation_file'] = name
    df['epoch_index'] = idx
    try:
        w = WCS(hdu.header)
        sky = w.pixel_to_world(df['xcentroid'].to_numpy(), df['ycentroid'].to_numpy())
        df['ra_deg'] = sky.ra.deg
        df['dec_deg'] = sky.dec.deg
    except Exception as e:
        print('WCS conversion failed for', name, ':', e)
        df['ra_deg'] = np.nan; df['dec_deg'] = np.nan
    tables.append(df)
    print(name, 'detected sources:', len(df), 'background median:', median, 'noise std:', std)
    fig, ax = plt.subplots(figsize=(7, 6))
    lo, hi = np.nanpercentile(img[np.isfinite(img)], [2, 99.5])
    ax.imshow(img, origin='lower', cmap='gray', vmin=lo, vmax=hi)
    ax.scatter(df['xcentroid'], df['ycentroid'], s=45, facecolors='none', edgecolors='red')
    ax.set_title('First-pass detections: ' + name, fontsize=8)
    ax.set_xlabel('Pixel x'); ax.set_ylabel('Pixel y'); plt.show()

if tables:
    detections = pd.concat(tables, ignore_index=True)
    detections.to_csv('spherex_first_pass_detections.csv', index=False)
    print('Saved spherex_first_pass_detections.csv')
    display(detections.head(20))
else:
    print('No source table produced. Do not interpret this as no astronomical sources; detector settings or image extension may need adjustment.')


## Important interpretation limits

- Do not compare raw pixel `(x,y)` values across the two cutouts until they are registered onto a common grid.
- Confirm the correct WCS is used. Some SPHEREx products may store spatial/spectral coordinate information in separate extensions or headers.
- Apply quality masks from `FLAGS` and account for `VARIANCE` before accepting detections.
- A source detected in one image but not the other is not automatically moving; it may be below threshold, masked, blended, or affected by artifacts.
- Only call an object a candidate after cross-matching, uncertainty/significance checks and independent validation.
